# Clinic Administrator Demo (draft)

Interactive walkthrough of **Astra** — the AI clinic administrator — using the **live Postgres database** from this repo.

**What this notebook shows**
- End-to-end **user journey**: onboarding → procedure consultation → schedule → registration → booking → verify → cancel
- Reads/writes the same `medical.*` schema the Telegram bot uses
- **Section 3** — real agent turn (LLM + `rag` tool); other steps call the same **services/tools** directly so each layer is visible

**What is intentionally omitted** (production-only)
- Guardrails classifiers
- Redis job queue & Telegram ingress
- Verbose tool-call JSON traces in the output

In [122]:
from __future__ import annotations

import asyncio
import json
from datetime import datetime, timedelta

import nest_asyncio
from sqlalchemy import text

nest_asyncio.apply()

import config
from db import dispose_engine, get_engine
from dialog_agent import DialogAgent
from dialog_agent.user_session import UserSessionStore
from guardrails.schemas import InputGuardrailVerdict, OutputGuardrailVerdict
from tools.book_appointment.service import BookAppointmentService
from tools.cancel_appointment.service import CancelAppointmentService
from tools.client_register.service import ClientRegisterService
from tools.get_appointment.service import GetAppointmentService
from tools.procedure_profile import PROCEDURE_PROFILES_CTX_KEY
from tools.schedule_check.service import ScheduleCheckService

In [123]:
# Jupyter cells are synchronous; services are async — this bridge matches production calls.
def run_async(coro):
    """Run async code in Jupyter."""
    return asyncio.get_event_loop().run_until_complete(coro)

In [124]:
# Demo user — DEMO_USER_ID is the session key (same role as telegram user id in worker).
DEMO_USER_ID = 9900001
DEMO_FULL_NAME = "Daniella Shlomi"
DEMO_PHONE = "+972501234567"
DEMO_BIRTH_DATE = "15.05.1990"
DEMO_EMAIL = "demo.client@example.com"
DEMO_PROCEDURE_QUERY = (
    "Hi! I'm thinking about botox for my forehead lines — "
    "how much would it cost and roughly how long is the appointment?"
)

## 1. Database snapshot

Quick peek at what the administrator can work with (no ETL — assumes data is already loaded).

In [125]:
# Define snapshot reader once; same schema the worker uses at runtime.
async def print_database_snapshot() -> None:
    """Print table counts and a sample of cheapest procedures."""
    table_queries = {
        "procedures": f"SELECT COUNT(*) FROM {config.DB_SCHEMA}.procedures",
        "doctors": f"SELECT COUNT(*) FROM {config.DB_SCHEMA}.doctors",
        "clients": f"SELECT COUNT(*) FROM {config.DB_SCHEMA}.clients",
        "appointments": f"SELECT COUNT(*) FROM {config.DB_SCHEMA}.appointments",
    }
    async with get_engine().connect() as conn:
        for table_name, query in table_queries.items():
            row_count = (await conn.execute(text(query))).scalar_one()
            print(f"{table_name}: {row_count}")

        sample_result = await conn.execute(
            text(
                f"""
                SELECT doctor, procedure, cost, anestesia
                FROM {config.DB_SCHEMA}.procedures
                ORDER BY cost ASC
                LIMIT 5
                """
            )
        )
        print("\nSample procedures (prices in NIS):")
        for procedure_row in sample_result.fetchall():
            print(
                f"  • {procedure_row.procedure} — Dr {procedure_row.doctor}, "
                f"{procedure_row.cost} NIS, anesthesia: {procedure_row.anestesia}"
            )

In [126]:
# Sanity check: DB reachable and seeded before simulating the client journey.
run_async(print_database_snapshot())

procedures: 20
doctors: 5
clients: 2
appointments: 2

Sample procedures (prices in NIS):
  • Lip augmentation consultation — Dr Smith E.M., 200 NIS, anesthesia: none
  • LED light therapy — Dr Davis L.Yu., 250 NIS, anesthesia: none
  • Gummy smile correction — Dr Johnson P.N., 300 NIS, anesthesia: none
  • Light chemical peel — Dr Smith E.M., 350 NIS, anesthesia: none
  • Laser hair removal (underarms) — Dr Williams A.A., 400 NIS, anesthesia: none


## 2. Onboarding — name and phone

Before the agent runs, the worker collects a verified profile (same as Telegram `/start` flow).

In [127]:
# Same as /start in beauty_worker: clear in-memory state for a fresh demo run.
session_store = UserSessionStore()
session_store.reset(DEMO_USER_ID)

UserSession(full_name=None, phone=None, onboarding_step=<OnboardingStep.AWAITING_NAME: 'awaiting_name'>, registration_step=None, birth_date=None, email=None, birth_date_attempts=0, email_attempts=0, pending_registration_prompt=False, is_registered=False, client_id=None)

In [128]:
# Onboarding step 1: validate full name (first + last) before agent access.
name_step = session_store.handle_onboarding(DEMO_USER_ID, DEMO_FULL_NAME)
print("Bot:", name_step.reply or "(name accepted)")

Bot: Thank you! Please enter your phone number (e.g. +79991234567 or +972501234567).


In [129]:
# Onboarding step 2: phone completes verified profile injected into every agent turn.
phone_step = session_store.handle_onboarding(DEMO_USER_ID, DEMO_PHONE)
print("Bot:", phone_step.reply)

user_session = session_store.get(DEMO_USER_ID)
verified_profile = user_session.to_profile()
print("\nVerified profile:", verified_profile)

Bot: Thank you! How can I help you today — procedure information or booking an appointment?

Verified profile: UserProfile(full_name='Daniella Shlomi', phone='+972501234567', is_registered=False, client_id=None)


## 3. Procedure consultation (agent + rag)

Real **DialogAgent** turn: client message → **rag** tool → LLM writes the reply.
Procedure profiles land in workflow context for booking.

In [130]:
# Production agent stack — guardrails disabled in notebook only (worker always runs them).
dialog_agent = DialogAgent(session_store)


async def _allow_input(*args, **kwargs) -> InputGuardrailVerdict:
    return InputGuardrailVerdict(allowed=True)


async def _allow_output(*args, **kwargs) -> OutputGuardrailVerdict:
    return OutputGuardrailVerdict(allowed=True)


dialog_agent._guardrails.check_input = _allow_input
dialog_agent._guardrails.check_output = _allow_output

# Same as /start in worker: drop in-process session + Postgres chat history for demo user.
run_async(dialog_agent.reset(DEMO_USER_ID))


async def ask_agent(user_text: str) -> str:
    """One dialog turn: tools (rag, etc.) + LLM client-facing reply."""
    current_session = session_store.get(DEMO_USER_ID)
    return await dialog_agent.handle(
        user_text,
        DEMO_USER_ID,
        profile=current_session.to_profile(),
    )


async def get_procedure_profiles_from_context() -> list[dict]:
    """Named profiles stored by rag tool — used by book_appointment in production."""
    session = dialog_agent._session(DEMO_USER_ID)
    stored = await session.context.store.get(PROCEDURE_PROFILES_CTX_KEY, default={})
    return list(stored.values())

In [131]:
# Client question in natural language — NOT SQL. Agent calls rag internally.
client_reply = run_async(ask_agent(DEMO_PROCEDURE_QUERY))
procedure_profiles = run_async(get_procedure_profiles_from_context())

print("Client question:", DEMO_PROCEDURE_QUERY)
print("\n--- Astra ---")
print(client_reply)
print("\n--- rag stored in context (internal, for booking demo) ---")
print(f"{len(procedure_profiles)} profile(s)")
for profile in procedure_profiles:
    print(profile)

Client question: Hi! I'm thinking about botox for my forehead lines — how much would it cost and roughly how long is the appointment?

--- Astra ---
Hello Daniella! 

For Botox treatment for forehead lines, the appointment costs **650 NIS** and takes approximately **one hour**. 

The procedure is performed by Dr. Smith E.M. and requires mandatory anesthesia, which is included in the total cost of **1,050 NIS**.

Would you like to book an appointment for this treatment? If so, I can check available times with Dr. Smith.

--- rag stored in context (internal, for booking demo) ---
1 profile(s)
{'procedure_id': 1, 'doctor': 'Smith E.M.', 'procedure': 'Botox (forehead)', 'cost': 650, 'anestesia': 'mandatory', 'if_anestesia_cost': 1050, 'duration': 60}


In [132]:
# Booking steps below need procedure_id from rag context (same as book_appointment tool).
if not procedure_profiles:
    raise RuntimeError(
        "No procedure profiles in context after agent turn. "
        "Re-run section 3 setup cell (dialog_agent.reset) then the ask_agent cell."
    )

selected_procedure = procedure_profiles[0]
print("Selected for booking demo:", selected_procedure)

Selected for booking demo: {'procedure_id': 1, 'doctor': 'Smith E.M.', 'procedure': 'Botox (forehead)', 'cost': 650, 'anestesia': 'mandatory', 'if_anestesia_cost': 1050, 'duration': 60}


## 4. Doctor schedule

Direct call to `ScheduleCheckService` — same logic as the **schedule_check** tool.

In [133]:
selected_doctor = selected_procedure["doctor"]
booking_date = (datetime.now() + timedelta(days=1)).strftime("%d.%m.%Y")

schedule_service = ScheduleCheckService()
schedule_result = run_async(
    schedule_service.check_schedule(selected_doctor, booking_date)
)
print(json.dumps(schedule_result, indent=2, default=str))

{
  "success": true,
  "doctor": "Smith E.M.",
  "date": "27.06.2026",
  "total_slots": 24,
  "free_slots": 24,
  "busy_slots": 0,
  "slots": [
    {
      "time": "09:00",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "09:20",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "09:40",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "10:00",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "10:20",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "10:40",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "11:00",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "12:20",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "12:40",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "13:00",
      "status": "free",
  

In [134]:
# Pick first free slot for the booking demo.
free_slot_times = [
    slot["time"]
    for slot in schedule_result.get("slots", [])
    if slot["status"] == "free"
]
booking_time = free_slot_times[0] if free_slot_times else "09:00"

print(f"Doctor: {selected_doctor}")
print(f"Date: {booking_date}")
print(
    f"Free slots: {', '.join(free_slot_times[:8]) or '(none — using demo fallback 09:00)'}"
)
print(f"Selected time for demo: {booking_time}")

Doctor: Smith E.M.
Date: 27.06.2026
Free slots: 09:00, 09:20, 09:40, 10:00, 10:20, 10:40, 11:00, 12:20
Selected time for demo: 09:00


## 5. Clinic registration

Direct path through `UserSessionStore` + `ClientRegisterService` — same as after **client_register** in production.

In [135]:
register_service = ClientRegisterService()

existing_client = run_async(
    register_service.get_client_by_telegram_id(DEMO_USER_ID)
)
if existing_client is None:
    phone_lookup = run_async(register_service.check_client_exists(DEMO_PHONE))
    if phone_lookup.get("exists"):
        existing_client = phone_lookup["client"]

print("Existing client:", existing_client)

Existing client: None


In [136]:
if existing_client:
    demo_client_id = existing_client["id"]
    user_session.apply_client_record(existing_client)
    print(
        f"Already registered: {existing_client['full_name']} "
        f"(internal client_id={demo_client_id})"
    )
else:
    session_store.start_registration(DEMO_USER_ID, DEMO_FULL_NAME, DEMO_PHONE)
    print("Bot:", session_store.ASK_BIRTH_DATE)
    run_async(session_store.handle_registration(DEMO_USER_ID, DEMO_BIRTH_DATE))
    print("Bot:", session_store.ASK_EMAIL)
    registration_result = run_async(
        session_store.handle_registration(DEMO_USER_ID, DEMO_EMAIL)
    )
    print("Bot:", registration_result.user_reply)
    demo_client_id = user_session.client_id
    print(f"Registered with internal client_id={demo_client_id}")

Bot: To complete your clinic registration, please enter your date of birth (e.g. 15.05.1990 or 1990-05-15).
Bot: Please enter your email address (e.g. name@example.com).
Bot: Your clinic profile has been registered successfully.
Registered with internal client_id=11


## 6. Book appointment

Direct call to `BookAppointmentService` — same as the **book_appointment** tool.

In [137]:
client_wants_anesthesia = selected_procedure.get("anestesia") == "mandatory"
appointment_duration = int(selected_procedure["duration"])

book_service = BookAppointmentService()
book_result = run_async(
    book_service.book(
        client_id=demo_client_id,
        procedure_id=int(selected_procedure["procedure_id"]),
        date=booking_date,
        time=booking_time,
        client_want_anestesia=client_wants_anesthesia,
        duration=appointment_duration,
    )
)
print(json.dumps(book_result, indent=2, default=str))

{
  "success": true,
  "appointment_id": 10,
  "message": "Appointment created successfully",
  "updated": false
}


## 7. Verify upcoming appointments

Direct call to `GetAppointmentService` — same as the **get_appointment** tool.

In [138]:
appointment_service = GetAppointmentService()
appointments_response = run_async(
    appointment_service.get_appointment(demo_client_id)
)

print(json.dumps(appointments_response, indent=2, default=str))

{
  "success": true,
  "appointments": [
    {
      "id": 10,
      "client_id": 11,
      "date": "27.06.2026",
      "time": "09:00",
      "procedure_id": 1,
      "duration": 60,
      "cost": 1050,
      "anestesia": "mandatory",
      "anestesia_used": "yes",
      "client_name": "Daniella Shlomi",
      "procedure_name": "Botox (forehead)",
      "doctor": "Smith E.M."
    }
  ],
  "count": 1
}


## 8. Cancel appointment

Direct call to `CancelAppointmentService` — same as the **cancel_appointment** tool.
Cancels the booking from section 6 and frees schedule slots.

In [139]:
cancel_service = CancelAppointmentService()
cancel_result = run_async(
    cancel_service.cancel_appointment(
        client_id=demo_client_id,
        date=booking_date,
        time=booking_time,
        procedure_id=int(selected_procedure["procedure_id"]),
    )
)
print(json.dumps(cancel_result, indent=2, default=str))

{
  "success": true,
  "appointment_id": 10,
  "message": "Appointment on 27.06.2026 09:00 (procedure: Botox (forehead)) cancelled successfully"
}


In [140]:
# Confirm the appointment list is empty (or no longer includes the cancelled visit).
appointments_after_cancel = run_async(
    appointment_service.get_appointment(demo_client_id)
)
print(json.dumps(appointments_after_cancel, indent=2, default=str))

{
  "success": false,
  "error": "No appointments from today onward for client ID 11"
}


In [141]:
run_async(dispose_engine())